# SEDICI multi-label subject classification — Colab 2026
Edit only the variables in the next cell, then use **Runtime → Run all**.

In [ ]:
# User configuration
BASE_DIR = '/content/drive/MyDrive/A___Maestria_en_ID/Tareas_PLN/001_1_Clasificador_Materias_SEDICI_Texto_Completo'
META_CSV = '/content/drive/MyDrive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/SEDICIpoblacion.csv'
MAP_CSV = '/content/drive/MyDrive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Mapeo_SEDICI_Rafa_data-1758643353469.csv'
FULLTEXT_SOURCE = '/content/drive/MyDrive/A___Maestria_en_ID/Tareas_PLN/100_datos_thesis_maestria/Datos_SEDICI/SEDICI_FullText_TXT'
EXECUTION_PROFILE = 'smoke'  # smoke | 20k | full | custom
CONFIG_FILE = 'configs/sparse_2026.yaml'  # used only when profile='custom'
USE_DUMMY_DATA = True
REPOSITORY_URL = 'https://github.com/KharolusIII/institutional-repository-subject-classification.git'
GITHUB_TOKEN_SECRET_NAME = 'GITHUB_TOKEN_IR_SUBJECT_CLASSIFICATION'
RESUME_CACHES = True


## 00 Environment

In [ ]:
import os, platform, subprocess, sys
print(sys.version)
print(platform.platform())
print('COLAB:', 'google.colab' in sys.modules)

## 01 Install project

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    github_token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
except Exception as exc:
    raise RuntimeError(f'Cannot read Colab secret: {GITHUB_TOKEN_SECRET_NAME}. Grant this notebook access to it.') from exc
if not github_token:
    raise RuntimeError(f'Missing Colab secret: {GITHUB_TOKEN_SECRET_NAME}')
askpass_path = Path('/content/git_askpass_ir_subject_classification.sh')
askpass_path.write_text("#!/bin/sh\ncase \"$1\" in\n  *Username*) echo \"x-access-token\" ;;\n  *) echo \"$GITHUB_TOKEN_IR_SUBJECT_CLASSIFICATION\" ;;\nesac\n", encoding='utf-8')
askpass_path.chmod(0o700)
git_env = os.environ.copy()
git_env['GIT_ASKPASS'] = str(askpass_path)
git_env['GIT_ASKPASS_REQUIRE'] = 'force'
git_env['GIT_TERMINAL_PROMPT'] = '0'
git_env['GITHUB_TOKEN_IR_SUBJECT_CLASSIFICATION'] = github_token
def run_git(command, description):
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
    if result.returncode != 0:
        safe_error = (result.stderr or result.stdout or 'Unknown Git error').replace(github_token, '[REDACTED]')
        raise RuntimeError(f'{description} failed (git exit {result.returncode}):\n{safe_error[-4000:]}')
    return result
PROJECT_DIR = Path(BASE_DIR)
PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
try:
    run_git(['git', 'ls-remote', '--exit-code', REPOSITORY_URL, 'refs/heads/main'], 'Repository authentication check')
    if (PROJECT_DIR / '.git').exists():
        run_git(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], 'Repository update')
    elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f'BASE_DIR exists but is not a Git repository and is not empty: {PROJECT_DIR}')
    else:
        run_git(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], 'Repository clone')
finally:
    git_env.pop('GITHUB_TOKEN_IR_SUBJECT_CLASSIFICATION', None)
    github_token = None
    askpass_path.unlink(missing_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_DIR}[all]'], check=True)
os.chdir(PROJECT_DIR)


## 02 Mount Google Drive

In [ ]:
assert Path('/content/drive/MyDrive').exists(), 'Google Drive is not mounted'
print('Repository workspace:', PROJECT_DIR)
print('Historical data remain external to the repository:', Path(META_CSV).parent)

## 03 Configuration

In [ ]:
from ir_subject_classification.config import load_config
PROFILE_CONFIGS = {'smoke': 'configs/run_smoke_2026.yaml', '20k': 'configs/run_20k_2026.yaml', 'full': 'configs/run_full_corpus_2026.yaml'}
if USE_DUMMY_DATA:
    subprocess.run([sys.executable, 'scripts/generate_dummy_data.py'], check=True)
    CONFIG_FILE = 'configs/dummy.yaml'
elif EXECUTION_PROFILE != 'custom':
    CONFIG_FILE = PROFILE_CONFIGS[EXECUTION_PROFILE]
config = load_config(CONFIG_FILE)
if not USE_DUMMY_DATA:
    config['data'].update(metadata_csv=META_CSV, mapping_csv=MAP_CSV or None, fulltext_source=FULLTEXT_SOURCE or None)
    config['experiment']['output_dir'] = str(PROJECT_DIR / 'outputs')
config

## 04 Hardware report

In [ ]:
import os
print('CPU cores:', os.cpu_count())
try:
 import torch
 print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as exc: print('Torch unavailable:', exc)

## 05 Input validation

In [ ]:
metadata_path = Path(config['data']['metadata_csv'])
assert metadata_path.is_file(), f'Missing metadata CSV: {metadata_path}'
print('Metadata:', metadata_path, metadata_path.stat().st_size, 'bytes')
if not USE_DUMMY_DATA:
    mapping_path = Path(config['data']['mapping_csv'])
    fulltext_path = Path(config['data']['fulltext_source'])
    assert mapping_path.is_file(), f'Missing mapping CSV: {mapping_path}'
    assert fulltext_path.is_dir(), f'Missing fulltext directory: {fulltext_path}'
    txt_probe = next(fulltext_path.glob('*.txt'), None)
    assert txt_probe is not None, f'No TXT files found directly under: {fulltext_path}'
    print('Mapping:', mapping_path, mapping_path.stat().st_size, 'bytes')
    print('Fulltext directory:', fulltext_path)
    print('TXT probe:', txt_probe.name)

## 06 Mapping and coverage
The package maps bitstream IDs only when full text is not inline.

In [ ]:
import pandas as pd
preview = pd.read_csv(metadata_path, nrows=5)
display(preview)

## 07 Dataset construction

In [ ]:
from ir_subject_classification.pipeline import construct_dataset
dataset_preview, target_schema = construct_dataset(config)
display(target_schema.head(20)); print(dataset_preview.shape)

## 08 Language identification

In [ ]:
from ir_subject_classification.pipeline import add_language_columns
language_preview = add_language_columns(dataset_preview.head(100), config)
display(language_preview.filter(regex='language').head())

## 09 Language statistics

In [ ]:
display(language_preview['abstract_detected_language'].value_counts(dropna=False))
display(language_preview['fulltext_detected_language'].value_counts(dropna=False))

## 10 Dataset statistics

In [ ]:
from ir_subject_classification.reporting import dataset_statistics, text_statistics
display(dataset_statistics(dataset_preview.labels.tolist()))
display(text_statistics(dataset_preview))

## 11 Sampling and splitting
Executed inside the pipeline using fixed seeds and iterative multi-label stratification.

## 12 Sparse representations
BoW, TF-IDF, and legacy BM25 are fitted on train only.

## 13 Dense representations
SBERT/LaBSE utilities support legacy truncation, chunked mean, and length-weighted mean. Dense models are downloaded only when enabled.

## 14 Training

In [ ]:
from ir_subject_classification.pipeline import run_pipeline
RUN_DIR = run_pipeline(config)
print('Run directory:', RUN_DIR)

## 15 Validation results

In [ ]:
validation = pd.read_csv(Path(RUN_DIR) / 'results_validation.csv')
display(validation.head(20))

## 16 Threshold optimization

In [ ]:
import json
display(json.loads((Path(RUN_DIR) / 'thresholds.json').read_text()))

## 17 Final test

In [ ]:
display(pd.read_csv(Path(RUN_DIR) / 'results_test.csv'))

## 18 Per-label analysis

In [ ]:
display(pd.read_csv(Path(RUN_DIR) / 'per_label_test.csv').sort_values('f1').head(20))

## 19 Language analysis

In [ ]:
display(pd.read_csv(Path(RUN_DIR) / 'language_agreement.csv'))

## 20 Error analysis

In [ ]:
pred_path = Path(RUN_DIR) / 'predictions_test.parquet'
predictions = pd.read_parquet(pred_path) if pred_path.exists() else pd.read_csv(Path(RUN_DIR) / 'predictions_test.csv')
display(predictions.head(20))

## 21 Export artifacts

In [ ]:
artifacts = sorted(str(path.relative_to(RUN_DIR)) for path in Path(RUN_DIR).rglob('*') if path.is_file())
print('\n'.join(artifacts))
print('Caches are reused when their content/config key is unchanged:', RESUME_CACHES)
print('\n--- Last 80 log lines ---')
log_lines = (Path(RUN_DIR) / 'run.log').read_text(encoding='utf-8').splitlines()
print('\n'.join(log_lines[-80:]))
from google.colab import files
# Uncomment to download the complete log:
# files.download(str(Path(RUN_DIR) / 'run.log'))